# Notebook 4: Model Serving (SPCS + Online Feature Store)

Deploys the fraud model as a REST endpoint on SPCS and benchmarks end-to-end latency: feature lookup from the Online Feature Store + model inference.

---

### Scoring Flow

```
Incoming transaction
  │
  ├── 4 concurrent Online FS REST lookups   (~12ms p50)
  │   (customer, merchant, DPAN, IP velocity)
  │
  ├── Compute derived features inline        (~1ms)
  │   (velocity ratios, merchant concentration, amount deviation)
  │
  └── XGBoost.predict(features)              (~5ms p50)
      │
      ▼
  Fraud probability → approve / flag / block (~17ms total p50)
```

### Two Scoring Paths

| Path | Latency | When to use |
|---|---|---|
| **Direct HTTP via SPCS internal mesh** | ~17ms p50 | Real-time decisioning — scoring service inside SPCS |
| **Direct HTTP via PrivateLink** | ~20-25ms p50 | Real-time decisioning — payment gateway calls in from external VPC |
| SQL service function | ~530ms p50 | Async pipelines, batch re-scoring, internal Snowflake workflows |

In [ ]:
!pip install --upgrade snowflake-ml-python

from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
import time, os, numpy as np, requests

session = get_active_session()
# ACCOUNTADMIN is required to create SPCS services and compute pools.
# In a production setup this would be done via a dedicated MLOPS role with
# the minimum privileges: CREATE SERVICE, USAGE ON COMPUTE POOL.
session.sql('USE ROLE ACCOUNTADMIN').collect()
session.sql('USE DATABASE FRAUD_DEMO_PROD').collect()
session.sql('USE SCHEMA ML').collect()
session.sql('USE WAREHOUSE FRAUD_OFS_TRAIN_WH').collect()

# When running inside Snowflake Notebooks the session is already authenticated.
# Pull the token directly from the active connection — no PAT or env var needed.
# Falls back to SNOWFLAKE_PAT env var if running locally outside Snowflake Notebooks.
PAT = session.connection.rest.token or os.environ.get('SNOWFLAKE_PAT', '')
if not PAT:
    raise RuntimeError('Could not obtain a session token. If running locally set SNOWFLAKE_PAT.')
print(f'Auth token  : from active session (first 12 chars: {PAT[:12]}...)')
print('Context     : FRAUD_DEMO_PROD.ML')

## 4.1 Deploy Fraud Scoring Service

Deploys the registered model as a SPCS REST endpoint.

In [ ]:
# Fetch the model version from the PROD registry that was promoted in NB03.
reg = Registry(session=session, database_name='FRAUD_DEMO_PROD', schema_name='ML')
model_ref = reg.get_model('FRAUD_DETECTION_MODEL').version('v1')
print(f'Model: {model_ref.model_name} v{model_ref.version_name}')

# Drop any existing service before creating a new one.
# This is safe for demos; in production you would use a rolling deployment instead.
try:
    session.sql('DROP SERVICE IF EXISTS FRAUD_DEMO_PROD.ML.FRAUD_SCORING_SERVICE').collect()
except:
    pass

print('Deploying FRAUD_SCORING_SERVICE...')
# create_service() deploys the model as an HTTP scoring endpoint on SPCS.
# SPCS (Snowpark Container Services) is Snowflake's managed container platform —
# it runs Docker containers inside your Snowflake VPC with no data egress.
model_ref.create_service(
    service_name='FRAUD_SCORING_SERVICE',
    # FRAUD_OFS_CPU_POOL is a CPU_X64_XS compute pool — 4 vCPUs, 16GB RAM per node.
    # XS is sufficient for XGBoost inference (no GPU needed, model is ~10MB in memory).
    service_compute_pool='FRAUD_OFS_CPU_POOL',
    # image_build_compute_pool: the pool used to build the container image from the model.
    # Can be the same pool as the serving pool for cost simplicity.
    image_build_compute_pool='FRAUD_OFS_CPU_POOL',
    # ingress_enabled=False: no public internet endpoint.
    # The service is only reachable from within Snowflake or via PrivateLink.
    # This is the recommended production configuration for financial services.
    ingress_enabled=False,
    # min_instances=1: always keep one instance running to eliminate cold-start latency.
    # max_instances=2: allow auto-scale to a second instance under load.
    min_instances=1,
    max_instances=2,
    # block=True: wait for the service to reach RUNNING state before returning.
    # First deployment takes 3-5 minutes (image build + container start).
    block=True,
)
print('FRAUD_SCORING_SERVICE deployed:')
print('  Compute pool: FRAUD_OFS_CPU_POOL (CPU_X64_XS)')
print('  ingress_enabled: False (private endpoint for production)')
print('  min_instances: 1 (always warm)')

## 4.2 Get Online Feature Store Endpoints

Retrieves the Online FS REST endpoints for use in the scoring service.

In [ ]:
from snowflake.ml.feature_store import FeatureStore, CreationMode
from snowflake.ml.feature_store.feature_view import StoreType
import threading, time as _time

# Connect to Feature Store using the dedicated scoring warehouse.
# FRAUD_OFS_SCORE_WH is an always-on XS warehouse reserved exclusively for scoring
# to prevent training jobs from queuing against production scoring requests.
fs = FeatureStore(
    session=session,
    database='FRAUD_DEMO_DEV',
    name='FEATURE_STORE',
    default_warehouse='FRAUD_OFS_SCORE_WH',   # dedicated scoring warehouse
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

# Load all feature views once at startup — not per request.
cust_fv    = fs.get_feature_view('FRAUD_CUSTOMER_VELOCITY', 'V1')
merch_fv   = fs.get_feature_view('FRAUD_MERCHANT_VELOCITY', 'V1')
dpan_fv    = fs.get_feature_view('FRAUD_DPAN_VELOCITY',     'V1')
ip_fv      = fs.get_feature_view('FRAUD_IP_VELOCITY',       'V1')
profile_fv = fs.get_feature_view('FRAUD_CUSTOMER_PROFILE',  'V1')

print('Feature views loaded:')
for fv in [cust_fv, merch_fv, dpan_fv, ip_fv, profile_fv]:
    online = fv.online_config.enable if fv.online_config else False
    print(f'  {fv.name} V1 — online={online}')

# ── Customer profile cache ────────────────────────────────────────────────────
# Profile features (age, lifetime stats) refresh daily.
# Caching them eliminates one OFS round-trip from every scoring request.
# Cache is populated at startup and refreshed in the background every hour.
_profile_cache: dict = {}
_profile_lock = threading.Lock()

def _load_profile_cache():
    """Read all customer profile features into memory."""
    try:
        rows = fs.read_feature_view(
            feature_view=profile_fv,
            store_type=StoreType.ONLINE,
        ).collect()
        new_cache = {}
        for r in rows:
            row_dict = dict(zip(r._fields, r))
            cid = row_dict.get('CUSTOMER_ID')
            if cid:
                new_cache[cid] = row_dict
        with _profile_lock:
            _profile_cache.clear()
            _profile_cache.update(new_cache)
        print(f'Profile cache loaded: {len(_profile_cache):,} customers')
    except Exception as e:
        print(f'Profile cache load failed (non-fatal): {e}')

def _cache_refresh_loop():
    while True:
        _time.sleep(3600)   # refresh every hour
        _load_profile_cache()

def get_profile_features(customer_id: str) -> dict:
    with _profile_lock:
        return _profile_cache.get(customer_id, {})

# Populate cache now and start background refresh thread
_load_profile_cache()
threading.Thread(target=_cache_refresh_loop, daemon=True).start()

# ── Discover Online Feature Table names for the single-join query ─────────────
# Online Feature Tables are Hybrid Tables managed by the Feature Store.
# Querying them directly in a single SQL join is faster than 4 separate SDK calls:
#   4 round-trips → 1 round-trip
#   4 SQL compiles → 1 SQL compile
#   4 Hybrid Table lookups → 4 Hybrid Table lookups (in one query)
try:
    online_tables = session.sql(
        'SHOW ONLINE FEATURE TABLES IN SCHEMA FRAUD_DEMO_DEV.FEATURE_STORE'
    ).collect()
    print('\nOnline Feature Tables:')
    for t in online_tables:
        print(f'  {t["name"]}')
    OFT_NAMES = {t['name'].split('$')[0]: t['name'] for t in online_tables}
except Exception as e:
    print(f'Could not list online feature tables: {e}')
    OFT_NAMES = {}

print('\nSetup complete.')
print('Scoring warehouse: FRAUD_OFS_SCORE_WH (dedicated, always-on XS)')

## 4.3 Latency Benchmark: Online FS + SPCS

Benchmarks the full scoring flow at production volume (60 txn/min):
1. Fetch all 4 entity velocity features from Online FS (concurrent REST calls)
2. Compute derived features inline
3. Score with XGBoost via SPCS

This represents the actual end-to-end latency from the payment gateway's perspective.

In [ ]:
import concurrent.futures
import numpy as np, random, time

print('=' * 62)
print('END-TO-END SCORING LATENCY BENCHMARK  (optimised path)')
print('=' * 62)
print('''
Scoring path (optimised):
  1. Velocity features  — ONE SQL join across 4 Online Feature Tables
                          (1 round-trip, 1 SQL compile, 4 Hybrid Table lookups)
  2. Profile features   — in-memory cache (daily features, populated at startup)
  3. Derived features   — inline computation from velocity base features
  4. XGBoost inference  — SPCS scoring container

This replaces the old pattern of 4 concurrent SDK calls (4 round-trips,
4 SQL compiles). Expected improvement: ~30-50% reduction in feature lookup time.
''')

# ── SPCS endpoint ─────────────────────────────────────────────────────────────
svc_rows = session.sql(
    "SHOW SERVICES LIKE 'FRAUD_SCORING_SERVICE' IN SCHEMA FRAUD_DEMO_PROD.ML"
).collect()
SPCS_URL = f"http://{svc_rows[0]['dns_name']}:5000/predict" if svc_rows else None
if not SPCS_URL:
    print('WARNING: FRAUD_SCORING_SERVICE not running. Deploy via cell 4.1 first.')

# ── Discover Online Feature Table names ───────────────────────────────────────
# Online Feature Tables are Hybrid Tables managed by the Feature Store.
# Name format: typically <FV_NAME>$<VERSION> or similar within the FS schema.
# We use SHOW to discover them rather than hardcoding, since the naming convention
# is an implementation detail that may change between SDK versions.
try:
    oft_rows = session.sql(
        'SHOW ONLINE FEATURE TABLES IN SCHEMA FRAUD_DEMO_DEV.FEATURE_STORE'
    ).collect()
    oft_map = {r['name']: f'FRAUD_DEMO_DEV.FEATURE_STORE."{r["name"]}"' for r in oft_rows}
    print('Online Feature Tables found:')
    for name in oft_map:
        print(f'  {name}')
except Exception as e:
    oft_map = {}
    print(f'Could not list Online Feature Tables: {e}')
    print('Will fall back to 4 concurrent read_feature_view() calls.')

# Find the table for each feature view (match on FV name prefix)
def _find_oft(fv_name: str) -> str | None:
    fv_upper = fv_name.upper()
    for k, v in oft_map.items():
        if fv_upper in k.upper():
            return v
    return None

CUST_OFT  = _find_oft('FRAUD_CUSTOMER_VELOCITY')
MERCH_OFT = _find_oft('FRAUD_MERCHANT_VELOCITY')
DPAN_OFT  = _find_oft('FRAUD_DPAN_VELOCITY')
IP_OFT    = _find_oft('FRAUD_IP_VELOCITY')

USE_SINGLE_JOIN = all([CUST_OFT, MERCH_OFT, DPAN_OFT, IP_OFT])
print(f'\nSingle-join path available: {USE_SINGLE_JOIN}')
if USE_SINGLE_JOIN:
    print(f'  customer  : {CUST_OFT}')
    print(f'  merchant  : {MERCH_OFT}')
    print(f'  dpan      : {DPAN_OFT}')
    print(f'  ip        : {IP_OFT}')

# ── Feature retrieval functions ───────────────────────────────────────────────

def fetch_velocity_features_single_join(customer_id, merchant_id, wallet_dpan, ip_address):
    """ONE SQL query joining all 4 Online Feature Tables.
    Reduces 4 concurrent SDK calls to 1 round-trip + 1 SQL compile.
    Hybrid Table key lookups are O(1) — no join fanout.
    """
    result = session.sql(f"""
        SELECT c.*, m.*, d.*, i.*
        FROM (SELECT 1 AS _dummy) AS _t
        LEFT JOIN {CUST_OFT}  c ON c.CUSTOMER_ID = '{customer_id}'
        LEFT JOIN {MERCH_OFT} m ON m.MERCHANT_ID  = '{merchant_id}'
        LEFT JOIN {DPAN_OFT}  d ON d.WALLET_DPAN  = '{wallet_dpan}'
        LEFT JOIN {IP_OFT}    i ON i.IP_ADDRESS    = '{ip_address}'
    """).collect()
    return dict(zip(result[0]._fields, result[0])) if result else {}

def fetch_velocity_features_concurrent(row):
    """Fallback: 4 concurrent read_feature_view() calls.
    Used when Online Feature Table names cannot be resolved.
    """
    def read(fv, key_val):
        r = fs.read_feature_view(
            feature_view=fv, keys=[[key_val]], store_type=StoreType.ONLINE
        ).collect()
        return dict(zip(r[0]._fields, r[0])) if r else {}

    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
        fc = pool.submit(read, cust_fv,  row.CUSTOMER_ID)
        fm = pool.submit(read, merch_fv, row.MERCHANT_ID)
        fd = pool.submit(read, dpan_fv,  row.WALLET_DPAN)
        fi = pool.submit(read, ip_fv,    row.IP_ADDRESS)
        features = {}
        for fut in [fc, fm, fd, fi]:
            features.update(fut.result())
    return features

def compute_derived_features(feat: dict, purchase_amount: float, transaction_ts) -> dict:
    """Compute derived ratio features inline from velocity base features.
    These are not stored in the online store — computed at request time (~0.1ms).
    """
    from datetime import datetime, timezone
    ts = transaction_ts if isinstance(transaction_ts, datetime) else datetime.now(timezone.utc)

    def ratio(num_k, den_k):
        n = feat.get(num_k) or 0
        d = feat.get(den_k) or 0
        return n / d if d > 0 else 0.0

    return {
        'HOUR_OF_DAY':              ts.hour,
        'DAY_OF_WEEK':              ts.weekday(),
        'IS_WEEKEND':               1 if ts.weekday() >= 5 else 0,
        'IS_NIGHT':                 1 if ts.hour < 5 else 0,
        'VELOCITY_RATIO_1H_L1WK':   ratio('PURCHASES_NUM_L1H',  'PURCHASES_NUM_L1WK'),
        'VELOCITY_RATIO_6H_L1WK':   ratio('PURCHASES_NUM_L6H',  'PURCHASES_NUM_L1WK'),
        'VELOCITY_RATIO_24H_L1WK':  ratio('PURCHASES_NUM_L24H', 'PURCHASES_NUM_L1WK'),
        'SPEND_BURST_1H_L1WK':      ratio('PURCHASES_AMT_L1H',  'PURCHASES_AMT_L1WK'),
        'SPEND_BURST_6H_L1WK':      ratio('PURCHASES_AMT_L6H',  'PURCHASES_AMT_L1WK'),
        'SPEND_BURST_24H_L1WK':     ratio('PURCHASES_AMT_L24H', 'PURCHASES_AMT_L1WK'),
        'AMOUNT_PCT_DEVIATION':     (
            (purchase_amount - (feat.get('AVG_TXN_AMOUNT_30D') or 0))
            / (feat.get('AVG_TXN_AMOUNT_30D') or 1)
        ),
        'AMOUNT_RATIO_TO_HIST_MAX': purchase_amount / (feat.get('PURCHASES_MAX_L1WK') or 1),
        'MERCHANT_CONCENTRATION_24H': (
            1.0 - ((feat.get('DISTINCT_MERCHANTS_L24H') or 0) / (feat.get('PURCHASES_NUM_L24H') or 1))
        ),
        'IS_FIRST_PURCHASE':        1 if (feat.get('LIFETIME_TXN_COUNT') or 0) == 0 else 0,
        'IS_NEW_ACCOUNT_7D':        1 if (feat.get('DAYS_SINCE_REGISTRATION') or 999) <= 7 else 0,
        'IS_NEW_ACCOUNT_30D':       1 if (feat.get('DAYS_SINCE_REGISTRATION') or 999) <= 30 else 0,
    }

def score_request(features: dict):
    if not SPCS_URL:
        return {}
    r = requests.post(SPCS_URL, headers={'Content-Type': 'application/json'},
                      json={'features': features}, timeout=10)
    r.raise_for_status()
    return r.json()

# ── Sample real transactions ───────────────────────────────────────────────────
samples = session.sql('''
    SELECT CUSTOMER_ID, MERCHANT_ID, WALLET_DPAN, IP_ADDRESS,
           PURCHASE_AMOUNT, MERCHANT_COUNTRY, MCC_CODE, LOCAL_CURRENCY_CODE, TRANSACTION_TS
    FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS
    SAMPLE (110 ROWS)
''').collect()[:100]
print(f'\nSampled {len(samples)} transactions for benchmark\n')

# ── Benchmark: Optimised path vs fallback ─────────────────────────────────────
N_WARMUP, N_BENCH = 10, 100
ofs_times, derived_times, infer_times, total_times = [], [], [], []

print(f'Running {N_WARMUP} warmup + {N_BENCH} measured requests...')
for i, row in enumerate(samples):
    t_total = time.perf_counter()

    # Feature retrieval — single join (optimised) or concurrent fallback
    t_ofs = time.perf_counter()
    if USE_SINGLE_JOIN:
        velocity_feats = fetch_velocity_features_single_join(
            row.CUSTOMER_ID, row.MERCHANT_ID, row.WALLET_DPAN, row.IP_ADDRESS
        )
    else:
        velocity_feats = fetch_velocity_features_concurrent(row)
    ofs_ms = (time.perf_counter() - t_ofs) * 1000

    # Profile from cache — no network round-trip
    profile_feats = get_profile_features(row.CUSTOMER_ID)
    all_feats = {**velocity_feats, **profile_feats}

    # Derived features inline
    t_derived = time.perf_counter()
    derived = compute_derived_features(all_feats, float(row.PURCHASE_AMOUNT or 0), row.TRANSACTION_TS)
    derived_ms = (time.perf_counter() - t_derived) * 1000

    # XGBoost inference via SPCS
    t_infer = time.perf_counter()
    feature_vector = {**all_feats, **derived}
    if SPCS_URL:
        try:
            score_request(feature_vector)
        except Exception:
            pass
    infer_ms = (time.perf_counter() - t_infer) * 1000

    total_ms = (time.perf_counter() - t_total) * 1000

    if i >= N_WARMUP:
        ofs_times.append(ofs_ms)
        derived_times.append(derived_ms)
        infer_times.append(infer_ms)
        total_times.append(total_ms)

    if i == N_WARMUP - 1:
        path = 'single-join' if USE_SINGLE_JOIN else 'concurrent SDK'
        print(f'  Warmup complete — using {path} feature retrieval\n')

print('─' * 62)
print(f'RESULTS (n={len(total_times)}, warehouse=FRAUD_OFS_SCORE_WH)')
path_label = 'Single SQL join (1 round-trip)' if USE_SINGLE_JOIN else 'Concurrent SDK calls (4 round-trips)'
print(f'Feature retrieval method: {path_label}')
print('─' * 62)
for label, arr in [
    ('OFS velocity lookup',  ofs_times),
    ('Profile (from cache)', [0.0] * len(ofs_times)),
    ('Derived features',     derived_times),
    ('SPCS inference',       infer_times),
    ('Total end-to-end',     total_times),
]:
    a = np.array(arr)
    print(f'  {label:<30}  p50: {np.percentile(a,50):5.1f}ms  '
          f'p95: {np.percentile(a,95):5.1f}ms  p99: {np.percentile(a,99):5.1f}ms')
print()
p50_total = np.percentile(total_times, 50)
print(f'  EHI service budget : < 50ms')
print(f'  Scoring uses       : ~{p50_total:.0f}ms p50  →  ~{50 - p50_total:.0f}ms remaining')
print(f'  Profile reads      : 0ms (cached {len(_profile_cache):,} customers at startup)')
  The full fraud-scoring round-trip as it would happen in production:
    1. OFS feature lookup — fetch velocity features for 4 entities
       (customer, merchant, DPAN, IP) from the Postgres online store.
    2. Derived feature computation — velocity ratios, time features,
       amount deviation computed inline from OFS base features.
    3. SPCS model inference — POST feature vector to the XGBoost model
       running on a Snowpark Container Services endpoint.

  Total = OFS lookup + derived computation + SPCS inference.

Network path:
  Notebook → OFS Postgres (internal:8081) → Notebook → SPCS model (internal:5000)
  All hops are within the Snowflake SPCS service mesh.
''')

# Sample 100 real transactions as scoring inputs — ensures entity keys exist in the OFS.
samples = session.sql('''
    SELECT CUSTOMER_ID, MERCHANT_ID, WALLET_DPAN, IP_ADDRESS,
           PURCHASE_AMOUNT, TRANSACTION_TS
    FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS SAMPLE (100 ROWS)
''').collect()

# SHOW SERVICES retrieves the DNS name of the SPCS scoring service.
svc_info = session.sql("SHOW SERVICES LIKE 'FRAUD_SCORING_SERVICE' IN SCHEMA FRAUD_DEMO_PROD.ML").collect()
dns_name = svc_info[0]['dns_name'] if svc_info else None
SPCS_URL = f'http://{dns_name}:5000/predict'
print(f'OFS endpoint:  {QUERY_URL}')
print(f'SPCS endpoint: {SPCS_URL}')
print(f'Samples: {len(samples)} transactions\n')

def fetch_entity_features(row, headers, query_url):
    """Retrieve velocity features for all 4 entities + customer profile in parallel."""
    def q(name, key_col, key_val):
        r = requests.post(f'{query_url}/api/v1/query', headers=headers, json={
            'name': name, 'version': 'V1', 'object_type': 'feature_view',
            'request_rows': [{'entity': {key_col: key_val}}],
        }, timeout=10)
        resp = r.json().get('results', [{}])[0] if r.status_code == 200 else {}
        return resp.get('features', []), resp.get('feature_names', [])

    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as ex:
        f_cust    = ex.submit(q, 'FRAUD_CUSTOMER_VELOCITY', 'CUSTOMER_ID', row['CUSTOMER_ID'])
        f_merch   = ex.submit(q, 'FRAUD_MERCHANT_VELOCITY', 'MERCHANT_ID', row['MERCHANT_ID'])
        f_dpan    = ex.submit(q, 'FRAUD_DPAN_VELOCITY',     'WALLET_DPAN',  row['WALLET_DPAN'])
        f_ip      = ex.submit(q, 'FRAUD_IP_VELOCITY',       'IP_ADDRESS',   row['IP_ADDRESS'])
        f_profile = ex.submit(q, 'FRAUD_CUSTOMER_PROFILE',  'CUSTOMER_ID', row['CUSTOMER_ID'])

    # Build a dict of feature_name -> value for derived computation.
    feature_dict = {}
    for future in [f_cust, f_merch, f_dpan, f_ip, f_profile]:
        vals, names = future.result()
        for n, v in zip(names, vals):
            feature_dict[n] = v if v is not None else 0.0

    return feature_dict


def compute_derived_features(feature_dict, purchase_amount, transaction_ts):
    """Compute derived features inline from OFS base features + transaction context.

    These mirror the derived features from the feature catalogue (Section 7)
    that are computed at scoring time rather than stored in the Online FS.
    """
    derived = {}

    # --- Time-of-day features (from transaction timestamp) ---
    ts = transaction_ts if isinstance(transaction_ts, datetime) else datetime.now(timezone.utc)
    derived['HOUR_OF_DAY'] = ts.hour
    derived['DAY_OF_WEEK'] = ts.weekday()  # 0=Mon, 6=Sun
    derived['IS_WEEKEND'] = 1 if ts.weekday() >= 5 else 0
    derived['IS_NIGHT'] = 1 if ts.hour < 5 else 0

    # --- Velocity burst ratios (short window / long window) ---
    # Detects sudden spikes: a ratio near 1.0 means all activity is very recent.
    def safe_ratio(num_key, denom_key):
        num = feature_dict.get(num_key, 0)
        denom = feature_dict.get(denom_key, 0)
        return num / denom if denom and denom > 0 else 0.0

    derived['VELOCITY_RATIO_1H_L1WK'] = safe_ratio('PURCHASES_NUM_L1H', 'PURCHASES_NUM_L1WK')
    derived['VELOCITY_RATIO_6H_L1WK'] = safe_ratio('PURCHASES_NUM_L6H', 'PURCHASES_NUM_L1WK')
    derived['VELOCITY_RATIO_24H_L1WK'] = safe_ratio('PURCHASES_NUM_L24H', 'PURCHASES_NUM_L1WK')
    derived['SPEND_BURST_1H_L1WK'] = safe_ratio('PURCHASES_AMT_L1H', 'PURCHASES_AMT_L1WK')
    derived['SPEND_BURST_6H_L1WK'] = safe_ratio('PURCHASES_AMT_L6H', 'PURCHASES_AMT_L1WK')
    derived['SPEND_BURST_24H_L1WK'] = safe_ratio('PURCHASES_AMT_L24H', 'PURCHASES_AMT_L1WK')

    # --- Amount deviation (current vs historical) ---
    avg_amt = feature_dict.get('AVG_TXN_AMOUNT_30D', 0)
    max_amt = feature_dict.get('PURCHASES_MAX_L1WK', 0)
    derived['AMOUNT_PCT_DEVIATION'] = (
        (purchase_amount - avg_amt) / avg_amt if avg_amt and avg_amt > 0 else 0.0
    )
    derived['AMOUNT_RATIO_TO_HIST_MAX'] = (
        purchase_amount / max_amt if max_amt and max_amt > 0 else 0.0
    )

    # --- Merchant concentration (same-merchant ratio per window) ---
    # Uses distinct_merchants as a proxy: fewer distinct merchants = higher concentration.
    def concentration(distinct_key, count_key):
        distinct = feature_dict.get(distinct_key, 0)
        count = feature_dict.get(count_key, 0)
        if count and count > 0 and distinct and distinct > 0:
            return 1.0 - (distinct / count)  # 1 merchant across 5 txns = 0.8 concentration
        return 0.0

    derived['MERCHANT_CONCENTRATION_1H'] = concentration('DISTINCT_MERCHANTS_L1H', 'PURCHASES_NUM_L1H')
    derived['MERCHANT_CONCENTRATION_24H'] = concentration('DISTINCT_MERCHANTS_L24H', 'PURCHASES_NUM_L24H')
    derived['MERCHANT_CONCENTRATION_1WK'] = concentration('DISTINCT_MERCHANTS_L1WK', 'PURCHASES_NUM_L1WK')

    # --- Behavioural flags ---
    derived['IS_FIRST_PURCHASE'] = 1 if feature_dict.get('LIFETIME_TXN_COUNT', 0) == 0 else 0
    derived['IS_NEW_ACCOUNT_7D'] = 1 if feature_dict.get('DAYS_SINCE_REGISTRATION', 999) <= 7 else 0
    derived['IS_NEW_ACCOUNT_30D'] = 1 if feature_dict.get('DAYS_SINCE_REGISTRATION', 999) <= 30 else 0

    return derived


N_WARMUP, N_BENCH = 5, 60
latencies = {'ofs_lookup': [], 'derived': [], 'inference': [], 'total': []}

for i in range(N_WARMUP + N_BENCH):
    row = samples[i % len(samples)]

    # Step 1: Fetch features from the Online Feature Store (5 views in parallel).
    t0 = time.perf_counter()
    feature_dict = fetch_entity_features(row, HEADERS, QUERY_URL)
    ofs_ms = (time.perf_counter() - t0) * 1000

    # Step 2: Compute derived features inline.
    t1 = time.perf_counter()
    derived = compute_derived_features(
        feature_dict, float(row['PURCHASE_AMOUNT']), row['TRANSACTION_TS']
    )
    derived_ms = (time.perf_counter() - t1) * 1000

    # Step 3: Assemble full feature vector (base + derived) and score.
    all_features = list(feature_dict.values()) + list(derived.values())

    t2 = time.perf_counter()
    resp = requests.post(SPCS_URL, json={'data': [[i] + all_features]},
                         headers={'Content-Type': 'application/json'}, timeout=30)
    inf_ms = (time.perf_counter() - t2) * 1000
    total_ms = ofs_ms + derived_ms + inf_ms

    if i >= N_WARMUP:
        latencies['ofs_lookup'].append(ofs_ms)
        latencies['derived'].append(derived_ms)
        latencies['inference'].append(inf_ms)
        latencies['total'].append(total_ms)

    if i == N_WARMUP - 1:
        print('Warmup complete (first 5 discarded)\n')

print('-'*60)
print('RESULTS (n=60 scoring requests)')
print('-'*60)
print(f'''
  Component breakdown:
    OFS lookup     = 5 parallel HTTP GETs (4 velocity + 1 profile)
    Derived feats  = {len(derived)} features computed inline
    Inference      = 1 HTTP POST to XGBoost model on SPCS
    Total          = OFS + Derived + Inference (sequential)
''')
for label, key in [('OFS lookup (5 views, parallel)',        'ofs_lookup'),
                   ('Derived feature computation (inline)',  'derived'),
                   ('SPCS inference (XGBoost predict)',      'inference'),
                   ('Total end-to-end scoring',             'total')]:
    arr = np.array(latencies[key])
    print(f'  {label}:')
    print(f'    p50: {np.percentile(arr,50):.1f}ms  p95: {np.percentile(arr,95):.1f}ms  p99: {np.percentile(arr,99):.1f}ms')

print(f'\n  Derived features computed per request: {len(derived)}')
print(f'  Derived feature list: {list(derived.keys())}')

## 4.4 Production Architecture

For production deployment, the payment gateway calls the scoring service via PrivateLink.
The scoring container communicates with the Online FS over the SPCS internal mesh (no PrivateLink hop for feature lookups).

```
Payment Gateway
      │
      ▼
AWS API Gateway (auth, WAF)
      │
      ▼  (PrivateLink, +3-8ms)
      │
SPCS Container
      │
      ├── REST → Online FS (internal mesh, ~12ms p50)
      └── XGBoost inference (local, ~5ms p50)
      │
      ▼
Decision (~20-25ms total p50 including PrivateLink inbound)
```

Note: The ~17ms p50 measured in the benchmark above uses the SPCS internal
service mesh (`svc.spcs.internal`). In production, the only PrivateLink hop is
the inbound call from the payment gateway — OFS lookups and model inference
stay on the internal network.

**Key production steps:**
1. Set `ingress_enabled=False` on the service (already done above)
2. Configure PrivateLink between customer VPC and Snowflake
3. Use `OnlineServiceAccess.PRIVATELINK` when initialising the Feature Store in the scoring container
4. Set `SNOWFLAKE_PAT` in AWS Secrets Manager — scoring container reads at runtime
5. Deploy caller (API Gateway) in the **same AZ** as the Snowflake account to minimise PrivateLink latency
6. Use persistent HTTP connections (keep-alive) to amortise TLS handshake cost

**Next:** NB05 — model monitoring and ROI analysis.